# CART: Dataset Integrity and Reproducibility Check

This notebook validates the datasets published in the CART repository.

It verifies:
- Dataset availability and readability.
- Compressed CSV integrity.
- Git LFS embedding availability.
- Dataset dimensions and file sizes.

The notebook does not modify the published datasets.

## 1. Repository Setup

In [1]:
import os
import subprocess
from pathlib import Path

repo_dir = Path("/content/cart")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/semoglou/cart.git", str(repo_dir)],
        check=True
    )
else:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", "main"],
        cwd=repo_dir,
        check=True
    )

os.chdir(repo_dir)

print("Repository ready:", repo_dir)

Repository ready: /content/cart


## 2. Retrieve Git LFS Datasets

In [2]:
import shutil
import subprocess
from pathlib import Path

repo_dir = Path("/content/cart")

# Install Git LFS if it is not available
if shutil.which("git-lfs") is None:
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "git-lfs"],
        check=True
    )

# Initialize Git LFS
subprocess.run(
    ["git", "lfs", "install"],
    cwd=repo_dir,
    check=True
)

# Download the published embedding arrays
subprocess.run(
    [
        "git", "lfs", "pull",
        "--include=dataset/2_screening_extraction/par_embeddings.npy,dataset/3_narrative_taxonomy/narrative_embeddings.npy"
    ],
    cwd=repo_dir,
    check=True
)

print("Git LFS datasets retrieved successfully.")

Git LFS datasets retrieved successfully.


## 3. Validate Published Datasets

In [3]:

from pathlib import Path
import pandas as pd
import numpy as np
import zipfile
import io

# Repository dataset directory
dataset_dir = Path("/content/cart/dataset")

results = []

def check_dataset(path):
    relative_path = path.relative_to(dataset_dir)

    result = {
        "Dataset": str(relative_path),
        "Size (MB)": round(path.stat().st_size / (1024 ** 2), 2),
        "Status": "OK",
        "Details": ""
    }

    try:
        # Detect empty files
        if path.stat().st_size == 0:
            raise ValueError("Empty file")

        # Detect Git LFS pointers
        with open(path, "rb") as f:
            header = f.read(150)

        if b"git-lfs.github.com/spec/v1" in header:
            raise ValueError("Git LFS pointer — actual file not downloaded")

        # ZIP archives containing CSV datasets
        if path.name.endswith(".csv.zip"):

            if not zipfile.is_zipfile(path):
                raise ValueError("Invalid ZIP archive")

            with zipfile.ZipFile(path) as archive:
                csv_files = [
                    name for name in archive.namelist()
                    if name.lower().endswith(".csv")
                ]

                if not csv_files:
                    raise ValueError("ZIP contains no CSV file")

                if archive.testzip() is not None:
                    raise ValueError("Corrupted ZIP member")

                with archive.open(csv_files[0]) as f:
                    df = pd.read_csv(f)

                result["Details"] = (
                    f"CSV readable | "
                    f"{df.shape[0]:,} rows × {df.shape[1]} columns"
                )

        # Standard CSV datasets
        elif path.suffix.lower() == ".csv":

            df = pd.read_csv(path)

            result["Details"] = (
                f"CSV readable | "
                f"{df.shape[0]:,} rows × {df.shape[1]} columns"
            )

        # NumPy embeddings
        elif path.suffix.lower() == ".npy":

            array = np.load(path, mmap_mode="r", allow_pickle=False)

            result["Details"] = (
                f"NumPy readable | "
                f"shape={array.shape} | dtype={array.dtype}"
            )

        # Text files
        elif path.suffix.lower() == ".txt":

            with open(path, "r", encoding="utf-8") as f:
                lines = [line for line in f if line.strip()]

            if not lines:
                raise ValueError("Text file contains no non-empty lines")

            result["Details"] = f"Text readable | {len(lines):,} non-empty lines"

        else:
            result["Status"] = "SKIPPED"
            result["Details"] = "Unsupported file format"

    except Exception as e:
        result["Status"] = "FAILED"
        result["Details"] = str(e)

    return result


print("CART — Published Dataset Integrity Check")
print("=" * 60)

if not dataset_dir.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {dataset_dir}")

dataset_files = sorted(
    path for path in dataset_dir.rglob("*")
    if path.is_file()
)

for path in dataset_files:
    results.append(check_dataset(path))

report = pd.DataFrame(results)

display(report)

print("\nSUMMARY")
print("=" * 60)

print("Total files:", len(report))
print("OK:", (report["Status"] == "OK").sum())
print("FAILED:", (report["Status"] == "FAILED").sum())
print("SKIPPED:", (report["Status"] == "SKIPPED").sum())

failed = report[report["Status"] == "FAILED"]

if failed.empty:
    print("\nAll published datasets passed the integrity checks.")
else:
    print("\nDatasets requiring attention:")
    display(failed[["Dataset", "Details"]])

CART — Published Dataset Integrity Check


,Dataset,Size (MB),Status,Details
0,1_preparation/01_paragraphs_clean.csv.zip,95.37,OK,"CSV readable | 117,716 rows × 8 columns"
1,1_preparation/01_stopwords_all.txt,0.00,OK,Text readable | 428 non-empty lines
2,1_preparation/05_Tempi_pargraphs_clean.csv.zip,60.15,OK,"CSV readable | 216,396 rows × 14 columns"
3,2_screening_extraction/02_cluster_metadata.csv,0.57,OK,CSV readable | 42 rows × 6 columns
4,2_screening_extraction/02_clusters_with_titles...,0.57,OK,CSV readable | 42 rows × 6 columns
5,2_screening_extraction/02_dataset_clustered.cs...,70.25,OK,"CSV readable | 82,401 rows × 10 columns"
6,2_screening_extraction/03_tempi_df_dataset.csv,30.95,OK,"CSV readable | 10,229 rows × 11 columns"
7,2_screening_extraction/04_tempi_df_R1_narrativ...,31.31,OK,"CSV readable | 10,229 rows × 12 columns"
8,2_screening_extraction/05_tempi_df_R1_extra_na...,3.10,OK,"CSV readable | 1,438 rows × 12 columns"
9,2_screening_extraction/05_tempi_df_R1_invalid_...,3.10,OK,"CSV readable | 1,438 rows × 12 columns"



SUMMARY
Total files: 14
OK: 14
FAILED: 0
SKIPPED: 0

All published datasets passed the integrity checks.
